In [35]:
import json
import os
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [36]:
import os
from pathlib import Path

# Root of the project. Defaults to the current working directory,
# or override via the RAG_BASE_DIR environment variable.
BASE_DIR = Path(os.getenv("RAG_BASE_DIR", Path.cwd()))
CHUNKS_DIR = str(BASE_DIR / "chunked")
OUTPUT_DIR = str(BASE_DIR / "VectorDB")
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

In [37]:
class MiniLMEmbeddings:
    def __init__(self, model_name="all-MiniLM-L6-v2", device=None):
        self.model = SentenceTransformer(model_name, device=device)

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )

In [38]:
def load_chunks(jsonl_path):
    chunks = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))
    return chunks

In [39]:
def is_valid_chunk(text):
    text = text.strip()

    if len(text) < 30:
        return False

    if re.fullmatch(r"[ivxlcdm]+", text.lower()):
        return False

    if re.fullmatch(r"\d+", text):
        return False

    return True

In [40]:
def prepare_chunks(chunks):
    texts = []
    metadatas = []

    for c in chunks:
        if not is_valid_chunk(c["text"]):
            continue

        texts.append(c["text"])
        metadatas.append({
            "chunk_id": c["metadata"]["chunk_id"],
            "pdf_name": c["metadata"]["pdf_name"],
            "page": c["metadata"]["page"]
        })

    return texts, metadatas

In [44]:
embed_model = MiniLMEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

In [42]:
FAISS_DIM = 384

faiss_index = faiss.IndexFlatIP(FAISS_DIM)

ALL_TEXTS = []
ALL_METADATAS = []

In [45]:
jsonl_files = [f for f in os.listdir(CHUNKS_DIR) if f.endswith(".jsonl")]

for file in jsonl_files:
    path = os.path.join(CHUNKS_DIR, file)
    print(f"Processing → {file}")

    chunks = load_chunks(path)
    texts, metadatas = prepare_chunks(chunks)

    if not texts:
        continue

    embeddings = embed_model.embed_documents(texts)

    faiss_index.add(embeddings)
    ALL_TEXTS.extend(texts)
    ALL_METADATAS.extend(metadatas)

print("Total vectors in FAISS:", faiss_index.ntotal)

Processing → common childhood_semantic_chunks.jsonl


Batches: 100%|█████████████████████████████| 12/12 [00:13<00:00,  1.14s/it]


Processing → Guidance for national_semantic_chunks.jsonl


Batches: 100%|███████████████████████████████| 9/9 [00:10<00:00,  1.20s/it]


Processing → Latent_semantic_chunks.jsonl


...

Processing → module6_semantic_chunks.jsonl


Batches: 100%|███████████████████████████████| 7/7 [00:07<00:00,  1.09s/it]


Processing → Recommendations_semantic_chunks.jsonl


Batches: 100%|███████████████████████████████| 4/4 [00:03<00:00,  1.03it/s]

Total vectors in FAISS: 3514


In [46]:
assert faiss_index.ntotal == len(ALL_TEXTS) == len(ALL_METADATAS)
print("Sanity check passed ✅")

Sanity check passed ✅


In [47]:
faiss_path = os.path.join(OUTPUT_DIR, "faiss.index")
faiss.write_index(faiss_index, faiss_path)

print("FAISS index saved at:", faiss_path)

FAISS index saved at: C:\Users\ritha\Desktop\AI-ML\ml projects\RAG 2.0\VectorDB\faiss.index


In [48]:
import pickle

with open(os.path.join(OUTPUT_DIR, "texts.pkl"), "wb") as f:
    pickle.dump(ALL_TEXTS, f)

with open(os.path.join(OUTPUT_DIR, "metadatas.pkl"), "wb") as f:
    pickle.dump(ALL_METADATAS, f)

print("Texts & metadata saved")

Texts & metadata saved
